# Market forecaster MAE and calibration

Measures the per-commodity forecast error of `PriceForecaster` across a small opponent pool and three horizons (1 step, 1 day, 5 days). Baseline for comparison is the naive constant-price forecast (predict `price(t)` for time `t+H`).

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from kaggle_environments import make

from kaggriculture.env.constants import PRODUCTS
from kaggriculture.env.observation import Observation
from kaggriculture.market.forecaster import PriceForecaster

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
BASE = "../src/kaggriculture/agent/baselines"
PAIRINGS = [
    ("v0-v0", f"{BASE}/v0_wheat.py", f"{BASE}/v0_wheat.py"),
    ("v3-v4", f"{BASE}/v3_market.py", f"{BASE}/v4_expansion.py"),
    ("starter-v3", "starter", f"{BASE}/v3_market.py"),
    ("v4-starter", f"{BASE}/v4_expansion.py", "starter"),
    ("random-v3", "random", f"{BASE}/v3_market.py"),
]
HORIZONS = [1, 24, 5 * 24]
SEEDS = (0, 7, 42)


def record_episode(agents, seed):
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed})
    env.run(list(agents))
    obs_list = []
    for step in env.steps:
        raw = dict(step[0].observation)
        raw.setdefault("player", 0)
        if "farms" not in raw:
            continue
        obs_list.append(Observation.from_dict(raw))
    return obs_list

In [ ]:
rows = []
predictions = []
for name, a, b in PAIRINGS:
    for seed in SEEDS:
        obs_list = record_episode((a, b), seed)
        fc = PriceForecaster(smoothing=0.2)
        for i, obs in enumerate(obs_list):
            fc.update(obs)
            for h in HORIZONS:
                if i + h >= len(obs_list):
                    continue
                pred = fc.predict(obs, horizon_steps=h)
                target = obs_list[i + h]
                for item in PRODUCTS:
                    rows.append(
                        {
                            "pairing": name,
                            "seed": seed,
                            "horizon": h,
                            "item": item,
                            "forecast": pred[item],
                            "naive": float(obs.market.prices[item]),
                            "actual": float(target.market.prices[item]),
                        }
                    )
df = pd.DataFrame(rows)
df["forecast_err"] = (df["forecast"] - df["actual"]).abs()
df["naive_err"] = (df["naive"] - df["actual"]).abs()
df.head()

In [ ]:
mae = (
    df.groupby(["horizon", "item"])[["forecast_err", "naive_err"]]
    .mean()
    .rename(columns={"forecast_err": "MAE_forecast", "naive_err": "MAE_naive"})
)
mae.round(2)

In [ ]:
totals = df.groupby("horizon")[["forecast_err", "naive_err"]].mean().round(3)
totals.columns = ["MAE_forecast", "MAE_naive"]
totals["lift_pct"] = (1 - totals["MAE_forecast"] / totals["MAE_naive"]) * 100
totals

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(4 * len(HORIZONS), 4), sharey=False)
for ax, h in zip(axes, HORIZONS, strict=False):
    sub = df[df["horizon"] == h]
    ax.scatter(sub["actual"], sub["forecast"], s=4, alpha=0.15, color="#1f77b4", label="forecast")
    ax.scatter(sub["actual"], sub["naive"], s=4, alpha=0.10, color="#d62728", label="naive")
    mx = max(sub["actual"].max(), sub["forecast"].max())
    ax.plot([0, mx], [0, mx], "k--", lw=0.7)
    ax.set_title(f"horizon = {h} steps")
    ax.set_xlabel("actual price ($)")
    ax.set_ylabel("predicted price ($)")
    ax.legend(loc="upper left", frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "forecaster-calibration.png", dpi=140, bbox_inches="tight")
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
wide = mae.reset_index().pivot(index="item", columns="horizon", values="MAE_forecast")
wide = wide.reindex(PRODUCTS)
x = np.arange(len(wide))
width = 0.25
for i, h in enumerate(HORIZONS):
    ax.bar(x + (i - 1) * width, wide[h], width, label=f"h={h} steps")
ax.set_xticks(x)
ax.set_xticklabels(wide.index, rotation=30, ha="right")
ax.set_ylabel("MAE ($)")
ax.set_title("Forecaster MAE per commodity, 3 seeds x 5 pairings")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "forecaster-mae-per-item.png", dpi=140, bbox_inches="tight")
fig.show()

## Notes

MELON MAE stays at zero because none of the baselines in this pool sell melon at any volume, so its market inventory sits at `I0 = 10000` and its price stays at base $250. The forecaster's rate on MELON is 0.0 throughout, so the forecast equals the current price which is also the true future price. A pairing with a melon-heavy agent would give a non-trivial number.

STRAWBERRY and MILK dominate the residual error. Both have `above_target >= 1.6` which means their price crashes fast under a glut. Predicting the timing of that crash from a smoothed rate is inherently noisy; the forecaster tends to underestimate crash speed.

At horizon 1 step the naive constant-price baseline is already tight (MAE < $1). The forecaster's marginal lift there is small in absolute terms; the value is at 24-step and longer horizons where the naive baseline degrades sharply.